# Estatística em campo · caderno da dupla
Extraia o pacote e abra este notebook ao lado da pasta data. Execute cada bloco quando ele aparecer no projetor. Use o kernel Python com as dependências de requirements.txt. Anote as conclusões na sua cópia.

## A · Conhecer o arquivo
Inspecione uma linha. Identifique a unidade observada e escolha uma coluna categórica e uma quantitativa.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display
ALPHA = 0.05
RAIZ = Path.cwd() if (Path.cwd() / 'data/penguins.csv').exists() else Path.cwd().parent
dados = pd.read_csv(RAIZ / 'data/penguins.csv')
display(dados.head())
print('Dimensão:', dados.shape)
display(dados.dtypes.to_frame('tipo'))
display(dados.isna().sum().to_frame('ausentes'))
display(dados.groupby('species').size().to_frame('registros'))
adelie = dados.loc[dados['species'].eq('Adelie')].copy()
print('Adelie:', len(adelie))
# Reproduza a seleção do atlas. Troque a ilha e o ano pela escolha da dupla.
print(dados.query("island == 'Dream' and year == 2007").species.value_counts())

## B · Descrever e decidir quem entra
Compare as medianas por espécie. Identifique um Adélie com massa conhecida e sexo ausente. Para qual pergunta ele contribui?

In [ ]:
display(dados.groupby('species')['body_mass_g'].describe())
y = adelie['body_mass_g'].dropna().to_numpy()
n = len(y)
print(f'Massa disponível: {n}/{len(adelie)}; média={y.mean():.2f} g; mediana={np.median(y):.2f} g; s={y.std(ddof=1):.2f} g')
fig, ax = plt.subplots(1, 2, figsize=(10,3))
ax[0].hist(y,bins=12,edgecolor='white'); ax[0].set(xlabel='Massa Adélie (g)',ylabel='Registros')
dados.boxplot(column='body_mass_g',by='species',ax=ax[1]); ax[1].set(ylabel='Massa (g)')
fig.suptitle(''); plt.tight_layout(); plt.show()
print(adelie.sex.value_counts(dropna=False))
display(adelie.loc[adelie.body_mass_g.notna() & adelie.sex.isna()])

## C · Massa média
Escreva H₀ e H₁ para a referência didática de 3700 g. Antecipe se ela está longe do centro observado. Execute e anote a conclusão sob o modelo, com IC95%, unidade e decisão a 5%.

In [ ]:
MU0 = 3700
media, s = y.mean(), y.std(ddof=1)
erro_padrao = s / np.sqrt(n)
t_calculado = (media - MU0) / erro_padrao
teste_media = stats.ttest_1samp(y, popmean=MU0)
ic_media = stats.t.interval(0.95, df=n-1, loc=media, scale=erro_padrao)
print(f'n={n}; média={media:.2f} g; diferença={media-MU0:.2f} g')
print(f't={t_calculado:.3f}; gl={n-1}; p={teste_media.pvalue:.4g}; IC95%={ic_media}')
print('Rejeitamos H0' if teste_media.pvalue < ALPHA else 'Não rejeitamos H0')

## D · Teste Z para uma proporção
Calcule a fração de fêmeas entre os Adélie com sexo registrado. Se a referência for 50%, quantos erros-padrão separam a fração observada dessa referência? Use as duas caudas da normal e α = 0,05. Discuta como uma coleta por casais afetaria a interpretação.

In [ ]:
sexo = adelie.loc[adelie['sex'].isin(['female','male']), 'sex']
n_sexo = len(sexo)
k = int(sexo.eq('female').sum())
p0 = 0.5
p_obs = k / n_sexo
ep0 = np.sqrt(p0 * (1 - p0) / n_sexo)
z = (p_obs - p0) / ep0
p_valor = 2 * stats.norm.sf(abs(z))  # Soma das duas caudas da normal padrão.

# IC de Wilson sem correção de continuidade, como no prop.test do R.
z_critico = stats.norm.isf(ALPHA / 2)
denominador = 1 + z_critico**2 / n_sexo
centro = (p_obs + z_critico**2 / (2 * n_sexo)) / denominador
margem = z_critico / denominador * np.sqrt(
    p_obs * (1 - p_obs) / n_sexo + (z_critico / (2 * n_sexo))**2
)
print(f'Fêmeas: {k}/{n_sexo} = {p_obs:.1%}; registros sem sexo: {len(adelie)-n_sexo}')
print(f'Z={z:.3f}; valor-p={p_valor:.4g}; IC95% de Wilson=[{centro-margem:.1%}, {centro+margem:.1%}]')
print('Rejeitamos H0' if p_valor < ALPHA else 'Não rejeitamos H0')

## E · Regressão
Antecipe o sinal da correlação. Compare retas pelo centro da nuvem e experimente outra inclinação. Qual reduz a soma dos quadrados dos resíduos?

In [ ]:
pares = adelie[['flipper_length_mm','body_mass_g']].dropna()
x = pares.flipper_length_mm
massa = pares.body_mass_g
for inclinacao in [0, 20, 33, 55]:
    previsto = massa.mean() + inclinacao * (x - x.mean())
    print(inclinacao, ((massa - previsto)**2).sum())

In [ ]:
pares = adelie[['flipper_length_mm','body_mass_g']].dropna()
x = pares['flipper_length_mm'].to_numpy()
y_reg = pares['body_mass_g'].to_numpy()
n_reg = len(pares)
print(f'Pares completos: {n_reg}/{len(adelie)}')
r = np.corrcoef(x,y_reg)[0,1]
ajuste = stats.linregress(x,y_reg)
predito = ajuste.intercept + ajuste.slope*x
residuos = y_reg - predito
critico = stats.t.ppf(0.975,df=n_reg-2)
ic_beta = (ajuste.slope-critico*ajuste.stderr, ajuste.slope+critico*ajuste.stderr)
print(f'r={r:.3f}; R²={r*r:.3f}')
print(f'Massa ajustada = {ajuste.intercept:.2f} + {ajuste.slope:.2f} × nadadeira')
print(f'Inclinação (g/mm): IC95%={ic_beta}; p={ajuste.pvalue:.4g}')
fig, ax = plt.subplots(1,3,figsize=(14,3.5))
ax[0].scatter(x,y_reg,alpha=.6)
ordem=np.argsort(x); ax[0].plot(x[ordem],predito[ordem],color='red')
ax[0].set(xlabel='Nadadeira (mm)',ylabel='Massa (g)')
ax[1].scatter(predito,residuos); ax[1].axhline(0,color='red'); ax[1].set(xlabel='Massa ajustada (g)',ylabel='Resíduo (g)')
stats.probplot(residuos,dist='norm',plot=ax[2]); plt.tight_layout(); plt.show()
print(f'Massa média ajustada para 190 mm: {ajuste.intercept+ajuste.slope*190:.2f} g')
print(f"Primeiro registro: observado {y_reg[0]:.0f} g, ajustado {predito[0]:.1f} g, resíduo {residuos[0]:+.1f} g")

Interprete a mudança estimada para mais 10 mm. Descreva uma característica do gráfico de resíduos.

## F · Caso resolvido em Sistemas de Informação
Acompanhe oito sessões fictícias, com um cliente distinto em cada sessão. Cada linha representa uma requisição sorteada. A quinta sessão tem 219 ms de latência e desfecho ausente.

In [ ]:
tamanho_kb = np.arange(10, 81, 10)
latencia_ms = np.array([189, 211, 202, 229, 219, 238, 237, 251])
sucesso = pd.Series([True, True, False, True, None, True, True, True], dtype="boolean")
print(f"{sucesso.sum()} sucessos em {sucesso.count()} desfechos conhecidos: {sucesso.mean():.1%}")

In [ ]:
teste_latencia = stats.ttest_1samp(latencia_ms, popmean=220)
ic_latencia = teste_latencia.confidence_interval(confidence_level=0.95)
reta_latencia = stats.linregress(tamanho_kb, latencia_ms)
print(f"Média = {latencia_ms.mean():.0f} ms, IC95% = [{ic_latencia.low:.2f}, {ic_latencia.high:.2f}] ms")
print(f"t = {teste_latencia.statistic:.3f}, gl = {teste_latencia.df}, valor-p = {teste_latencia.pvalue:.3f}")
print(f"r = {reta_latencia.rvalue:.3f}, intercepto = {reta_latencia.intercept:.2f} ms")
print(f"Inclinação = {reta_latencia.slope:.3f} ms/kB")

**Resultado.** A taxa de sucesso é 6/7 = 85,7%. As oito latências têm média de 222 ms, com IC95% de 204,72 a 239,28 ms. O teste t bilateral contra 220 ms dá valor-p de 0,792. A 5%, não rejeitamos a hipótese de média igual a 220 ms. A correlação é 0,934. Nos 10 a 80 kB observados, mais 10 kB correspondem a 7,88 ms a mais na latência média ajustada.